# EDA — Processed Bank Transactions

Exploratory analysis on the cleaned dataset produced by `etl-pipeline/transform.py`.

In [104]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# Plot styling
plt.style.use("ggplot")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 1. Load Processed Data

In [105]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

file = PROCESSED_DATA_DIR / "bank_transactions_cleaned.csv"

if not file.exists():
    raise FileNotFoundError(
        f"Processed file not found: {file}\n"
        "Run: python etl-pipeline/transform.py"
    )

df = pd.read_csv(file, parse_dates=["transaction_date"])

print(file.name)
print(f"Shape: {df.shape}")

bank_transactions_cleaned.csv
Shape: (6567, 17)


In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.columns.tolist()

In [ ]:
df.info()

## 2. Summary Stats

### Numerical

In [ ]:
df.describe().T

### Categoricals

In [ ]:
df.describe(include=["object", "string"]).T

## 3. Missing Values

After transform, nulls should be fully resolved.

In [ ]:
missing = (
    df.isnull()
      .sum()
      .to_frame("Missing")
)

missing["Percent"] = (
    missing["Missing"] / len(df) * 100
)

missing.sort_values("Missing", ascending=False)

In [ ]:
plt.figure(figsize=(10, 4))

sns.heatmap(
    df.isnull(),
    cbar=False,
    yticklabels=False
)

plt.title("Missing Values Heatmap (Processed)")
plt.show()

## 4. Duplicate Records

In [ ]:
duplicates = df.duplicated().sum()

print(f"Duplicate Rows: {duplicates}")

## 5. Case Inconsistency Check

After transform, categorical labels should be case-standardized (e.g. `Swansea` vs `swansea` should collapse to one value). This check validates that normalization worked.

In [ ]:
case_check_cols = [
    "transaction_type",
    "category",
    "location_city",
    "location_country",
]

case_collisions = []

for col in case_check_cols:
    values = df[col].dropna().astype(str).str.strip()
    groups = {}

    for value in values.unique():
        groups.setdefault(value.casefold(), set()).add(value)

    for key, variants in groups.items():
        if len(variants) > 1:
            case_collisions.append({
                "Column": col,
                "Canonical": key,
                "Variants": sorted(variants),
                "Variant Count": len(variants),
                "Row Count": int(values.str.casefold().eq(key).sum()),
            })

if case_collisions:
    case_df = pd.DataFrame(case_collisions)
    display(case_df)
else:
    print("No case-only duplicate labels found.")

## 6. Data Types

In [ ]:
df.dtypes

## 7. Unique Values

In [ ]:
unique = pd.DataFrame({
    "Unique Values": df.nunique(),
    "Data Type": df.dtypes
})

unique

## 8. Numerical Features

In [ ]:
numeric_cols = [
    "debit_amount",
    "credit_amount",
    "amount",
    "abs_amount",
    "balance",
]

numeric_cols

In [ ]:
for col in numeric_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[col], bins=40, kde=True)
    plt.title(f"Distribution — {col}")
    plt.show()

In [ ]:
for col in numeric_cols:
    plt.figure(figsize=(8, 3))
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot — {col}")
    plt.show()

## 9. Categorical Features

In [ ]:
categorical_cols = [
    "transaction_type",
    "transaction_direction",
    "category",
    "location_city",
    "location_country",
    "day_of_week",
]

categorical_cols

In [ ]:
for col in categorical_cols:
    plt.figure(figsize=(10, 4))

    (
        df[col]
        .value_counts()
        .head(15)
        .plot(kind="bar")
    )

    plt.title(col)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 10. Correlation Analysis

In [ ]:
corr = df[numeric_cols + ["year", "month"]].corr(numeric_only=True)

plt.figure(figsize=(10, 8))

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Matrix (Processed)")
plt.tight_layout()
plt.show()

## 11. Outlier Detection

In [ ]:
for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)

    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = df[
        (df[col] < lower) |
        (df[col] > upper)
    ]

    print(f"{col}: {len(outliers)} outliers")

## 12. Time-Series Trends

Uses derived date features from the transform step.

In [ ]:
print(f"Date range: {df['transaction_date'].min().date()} -> {df['transaction_date'].max().date()}")
print(f"Years covered: {sorted(df['year'].unique())}")

In [ ]:
monthly = (
    df.groupby("year_month", as_index=False)
      .agg(
          transactions=("transaction_number", "count"),
          total_debit=("debit_amount", "sum"),
          total_credit=("credit_amount", "sum"),
          avg_balance=("balance", "mean"),
      )
)

monthly.head()

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(monthly["year_month"], monthly["transactions"])
plt.title("Monthly Transaction Volume")
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(monthly["year_month"], monthly["total_debit"], label="Debit")
plt.plot(monthly["year_month"], monthly["total_credit"], label="Credit")
plt.title("Monthly Debit vs Credit Totals")
plt.xticks(rotation=90, fontsize=7)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(monthly["year_month"], monthly["avg_balance"])
plt.title("Average Monthly Balance")
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.show()

## 13. Spending & Income Patterns

In [ ]:
direction_summary = (
    df.groupby("transaction_direction")
      .agg(
          count=("transaction_number", "count"),
          total_abs_amount=("abs_amount", "sum"),
          avg_abs_amount=("abs_amount", "mean"),
      )
)

direction_summary

In [ ]:
category_spend = (
    df[df["transaction_direction"] == "Debit"]
      .groupby("category", as_index=False)
      .agg(
          transactions=("transaction_number", "count"),
          total_spent=("debit_amount", "sum"),
          avg_spent=("debit_amount", "mean"),
      )
      .sort_values("total_spent", ascending=False)
)

category_spend.head(15)

In [ ]:
plt.figure(figsize=(10, 5))

top_categories = category_spend.head(12)

sns.barplot(
    data=top_categories,
    x="total_spent",
    y="category",
)

plt.title("Top Categories by Total Debit Spend")
plt.tight_layout()
plt.show()

In [ ]:
dow_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday",
]

dow = (
    df.groupby("day_of_week", as_index=False)
      .agg(
          transactions=("transaction_number", "count"),
          total_abs_amount=("abs_amount", "sum"),
      )
)

dow["day_of_week"] = pd.Categorical(dow["day_of_week"], categories=dow_order, ordered=True)
dow = dow.sort_values("day_of_week")

plt.figure(figsize=(8, 4))
sns.barplot(data=dow, x="day_of_week", y="transactions")
plt.title("Transactions by Day of Week")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 14. Location Insights

In [ ]:
city_summary = (
    df.groupby("location_city", as_index=False)
      .agg(
          transactions=("transaction_number", "count"),
          total_abs_amount=("abs_amount", "sum"),
      )
      .sort_values("transactions", ascending=False)
)

city_summary.head(15)

In [ ]:
country_summary = (
    df.groupby("location_country", as_index=False)
      .agg(
          transactions=("transaction_number", "count"),
          total_abs_amount=("abs_amount", "sum"),
      )
      .sort_values("transactions", ascending=False)
)

country_summary

## Findings

- Processed data has no missing values and no duplicate rows.
- Case-only label collisions (e.g. `Swansea` vs `swansea`) are resolved after transform.
- Debit/credit sides are unified via `transaction_direction`, `amount`, and `abs_amount`.
- Derived date fields support monthly volume, spend, and balance trend analysis.
- Category and location fields are cleaned and ready for BI / load into the database.
- UK is the country with the highest no. of transactions while in terms of city, Swansea has the highest one. 
- There are more debit than credit transactions.
- Monday is the day of week with the highest volume of transactions.